In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/02_api_football_client

In [0]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide API settings, secret references, and raw storage paths are
# loaded from 00_project_config.
#
# HTTP communication, request pacing, rate-limit handling, and atomic JSON
# writing are provided by 02_api_football_client.

ENDPOINT = "leagues"

OUTPUT_FILE = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/competition_data/raw_competitions.json"
)


# -----------------------------------------------------------------------------
# Load API credential
# -----------------------------------------------------------------------------
# Credentials are retrieved from Databricks Secrets.
#
# API keys must never be hard-coded in notebooks or committed to source
# control.

api_key = dbutils.secrets.get(
    scope=API_FOOTBALL_SECRET_SCOPE,
    key=API_FOOTBALL_SECRET_KEY,
)


# -----------------------------------------------------------------------------
# Fetch competitions from API-Football
# -----------------------------------------------------------------------------
# The /leagues endpoint can return the complete competition catalogue in one
# request, so pagination is not required here.
#
# This is especially useful on the free API-Football plan because this
# ingestion consumes only one request from the daily allowance.
#
# The complete API payload is retained rather than extracting only "response".
# Keeping the raw response envelope preserves useful source metadata such as:
#
#   get
#   parameters
#   errors
#   results
#   paging
#   response
#
# Bronze can then decide which parts of the source structure to consume.

with APIFootballClient(
    api_key=api_key,
    base_url=API_FOOTBALL_BASE_URL,
    min_request_interval_seconds=API_FOOTBALL_MIN_REQUEST_INTERVAL_SECONDS,
) as client:

    payload = client.get(
        endpoint=ENDPOINT,
        params={},
    )

    # -------------------------------------------------------------------------
    # Validate expected source structure
    # -------------------------------------------------------------------------
    # The client already validates HTTP status codes and API-level errors.
    # Here we additionally verify the business payload expected by this
    # ingestion notebook.

    competitions = payload.get("response")

    if not isinstance(competitions, list):
        raise APIFootballError(
            "Expected API-Football 'response' to be a list "
            "for the leagues endpoint."
        )

    if not competitions:
        raise APIFootballError(
            "API-Football returned zero competitions. "
            "Raw output will not be overwritten."
        )

    # -------------------------------------------------------------------------
    # Persist raw source payload
    # -------------------------------------------------------------------------
    # write_json_atomic() writes to a temporary file first and replaces the
    # final file only after serialization succeeds.
    #
    # This protects the existing raw snapshot from being replaced by a
    # partially written file if the write fails.

    write_json_atomic(
        data=payload,
        file_path=OUTPUT_FILE,
    )

    # -------------------------------------------------------------------------
    # Completion summary
    # -------------------------------------------------------------------------

    print(
        f"Successfully ingested {len(competitions)} "
        "API-Football competitions."
    )

    print(f"Raw file written to: {OUTPUT_FILE}")

    print(
        "Daily API requests remaining: "
        f"{client.daily_remaining}"
    )